<h1 align="center">Rwanda Maize–Urea Trial</h1>
<h2 align="center">2. Data Cleaning & Integration</h2>

**Goal:** turn the three source tables into one analysis-ready dataset (one row per plot) **without silently changing or deleting anything**.

Rules:
* Source values are never overwritten. Corrections go in new columns and problems go in `flag_*` columns.
* Every join is validated (many plots → one field) and must keep all plots.
* No imputation at this stage.

The code lives in [`src/maize/data.py`](../src/maize/data.py), so the pipeline and the notebooks use identical logic.

> **Data note.** The trial data is confidential and is not in this repository. The outputs shown come from the **synthetic sample** in `01_Data/sample/`, which has the same structure and data-quality problems as the real data. The numbers illustrate the method; they are not Evergreen's actual results.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "02_Notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from maize import config, data

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
MODE = config.data_mode()
OUT = config.output_paths(MODE)
print("Data source:", "confidential trial data (01_Data/raw)" if MODE == "raw" else "synthetic sample (01_Data/sample)")

Data source: synthetic sample (01_Data/sample)


## 1. Clean each table

In [2]:
src = data.load_sources(MODE)
field = data.clean_field(src["field"])
plot = data.clean_plot(src["plot"])
soil = data.clean_soil(src["soil"])
print("Plot table:", src["plot"].shape, "->", plot.shape)

Plot table: (1590, 27) -> (1590, 48)


### Dates

The seven date columns arrive as text. Each gets a parsed `*_parsed` column. `clean_plot` raises an error if any non-missing value fails to parse, so a missing parsed date always means the source was blank.

In [3]:
pd.DataFrame({
    "missing in source": [plot[c].isna().sum() for c in config.DATE_COLUMNS],
    "missing after parsing": [plot[f"{c}_parsed"].isna().sum() for c in config.DATE_COLUMNS],
}, index=config.DATE_COLUMNS)

,missing in source,missing after parsing
planting_date,0,0
germ_count_date,0,0
urea_V6_date,0,0
urea_V10_date,1306,1306
silking_date,9,9
R3_date,4,4
harvest_date,0,0


### Data-quality flags

In [4]:
flags = [c for c in plot.columns if c.startswith("flag_")]
summary = plot[flags].sum().rename("plots flagged").to_frame()
summary.loc["urea total differs from planting + V6 + V10"] = (plot["urea_total_diff"] != 0).sum()
summary.loc["fields with implausible altitude"] = field["flag_implausible_altitude"].sum()
summary

,plots flagged
flag_germ_count_date_before_planting_date,0
flag_urea_V6_date_before_planting_date,0
flag_silking_date_before_planting_date,0
flag_R3_date_before_silking_date,3
flag_harvest_date_before_R3_date,0
flag_harvest_date_before_planting_date,0
flag_v10_date_missing,115
flag_implausible_yield,1
urea total differs from planting + V6 + V10,160
fields with implausible altitude,3


| Flag | Decision |
|---|---|
| R3 recorded before silking | Keep; exclude only from date-based variables. The true date is unknown |
| V10 dose with no V10 date | Keep; assume applied (quantity recorded) |
| Urea total off by ±1 kg/ha | Keep source total; `urea_total_calc` holds the recomputed value |
| Implausible yield | Keep; sensitivity analysis in notebooks 04 and 06 |
| Implausible altitude | `alt_clean` = missing; original `alt` kept |

## 2. Integrate

Plot ⟶ field on (`block_id`, `season`), validated as many-to-one, then ⟶ soil on `plot_id`, validated as one-to-one. Left joins keep every plot.

In [5]:
combined = data.integrate(field, plot, soil, data.load_external())
data.sanity_summary(combined)

,check,result
0,Rows (plots),1590
1,Columns,80
2,Duplicate plot_id,0
3,Missing yield,0
4,Missing treatment,0
5,Plots without field record,8
6,Plots with soil record,685
7,Plots in incomplete units,14
8,Implausible yields flagged,1


In [6]:
combined.groupby(
    "urea_rate_timing")["yield_tha"].size().reindex(
        config.TREATMENTS).rename(
            index=config.TREATMENT_LABELS).to_frame("plots")

,plots
urea_rate_timing,
"A: 50 kg/ha, all at V6",399
"B: 100 kg/ha, planting + V6",394
"C: 100 kg/ha, V6 + V10",399
"D: 100 kg/ha, all at V6",398


`has_soil_record` separates *no soil record* from *a soil record with missing pH*, which a missing pH alone cannot do.

## 3. Save

In [7]:
field.to_csv(OUT.processed / "field_clean.csv", index=False)
plot.to_csv(OUT.processed / "plot_clean.csv", index=False)
soil.to_csv(OUT.processed / "soil_clean.csv", index=False)
combined[data.public_columns(combined)].to_csv(OUT.processed / "combined_dataset.csv", index=False)
print("Saved to", OUT.processed.relative_to(config.ROOT))

Saved to 01_Data\processed


## Summary

* All plots are retained, with no duplicates and no missing yields or treatments.
* Labels are standardised (`crop`, `treat_details_std`) and dates parsed alongside the originals.
* Problems are flagged, not fixed by guesswork.
* The **combined dataset** (`combined_dataset.csv`, one row per plot) is brief deliverable 1. It stays local because it contains confidential data.

**Next:** notebook 03 explores the data.